In [89]:
!pip install docx2txt

In [90]:
# 2.2.1 데이터 리더
from llama_index.core import SimpleDirectoryReader

In [91]:
documents = SimpleDirectoryReader("sample_docs").load_data()

In [92]:
for i, document in enumerate(documents):
    print(f"Document {i+1}:")
    print(document.text)
    print("\n")

Document 1:
text file 1 txt


Document 2:
test file 2 docx


Document 3:
file3 hwp to pdf




In [93]:
documents = SimpleDirectoryReader("sample_docs", recursive=True).load_data()

In [94]:
for i, document in enumerate(documents):
    print(f"Document {i+1}:")
    print(document.text)
    print("\n")

Document 1:
text file 1 txt


Document 2:
test file 2 docx


Document 3:
file3 hwp to pdf


Document 4:
test file4 txt


Document 5:
test file5 md




In [95]:
# 특정 파일 형식만 로딩하기 P.42
documents = SimpleDirectoryReader(
    "sample_docs", 
    required_exts=[".txt", ".pdf"],
    recursive=True
).load_data()


In [96]:
for i, document in enumerate(documents):
    print(f"Document {i+1}:")
    print(document.text)
    print("\n")

Document 1:
text file 1 txt


Document 2:
file3 hwp to pdf


Document 3:
test file4 txt




In [97]:
# 문서 객체의 메타데이터 확인하기
import pprint
for i, document in enumerate(documents):
    print(f"Document {i+1} metadata:")
    pprint.pprint(document.metadata)
    print("\n")

Document 1 metadata:
{'creation_date': '2026-03-28',
 'file_name': 'file1.txt',
 'file_path': 'c:\\llamaindex\\ch02\\sample_docs\\file1.txt',
 'file_size': 15,
 'file_type': 'text/plain',
 'last_modified_date': '2026-03-29'}


Document 2 metadata:
{'creation_date': '2026-03-29',
 'file_name': 'file3.pdf',
 'file_path': 'c:\\llamaindex\\ch02\\sample_docs\\file3.pdf',
 'file_size': 10564,
 'file_type': 'application/pdf',
 'last_modified_date': '2026-03-29',
 'page_label': '1'}


Document 3 metadata:
{'creation_date': '2026-03-28',
 'file_name': 'file4.txt',
 'file_path': 'c:\\llamaindex\\ch02\\sample_docs\\sub_sample_docs\\file4.txt',
 'file_size': 14,
 'file_type': 'text/plain',
 'last_modified_date': '2026-03-29'}




In [98]:
# 2.2.2 데이터 커넥터 (외부 데이터 불러오기)
# 라마인덱스 내장 데이터 커넥터 한계 => 라마허브에서 제공하는 추가 커넥터 활용
# 라마허브: 데이터 커넥터 레지스트리 (다양한 외부 소스와 연동 가능한 커넥터들을 내려받아 사용 가능.)
# 데이터베이스 리더(DatabaseReader): SQL 데이터베이스에 쿼리 실행, 결과의 모든 행을 문서로 반환하는 커넥터.

In [99]:
!pip install llama-index-readers-database==0.3.0

In [100]:
!pip install pymysql

In [101]:
!pip install cryptography

In [102]:
!pip install --upgrade pip

In [103]:
!pip install cryptography

In [104]:
from sqlalchemy import create_engine
from llama_index.readers.database import DatabaseReader
from dotenv import load_dotenv
from cryptography.fernet import Fernet
import os

# .env 파일 로드 (override=True: 커널에 캐시된 이전 값 덮어쓰기)
load_dotenv(override=True)

enc_key = os.getenv("DB_ENC_KEY")
enc_password = os.getenv("DB_PASSWORD_ENC")


# Fernet으로 비밀번호 복호화
cipher = Fernet(enc_key)
db_password = cipher.decrypt(enc_password).decode()

# .env에서 나머지 DB 설정 읽기
scheme = "mysql+pymysql"
host   = os.getenv("DB_HOST")
port   = os.getenv("DB_PORT")
user   = os.getenv("DB_USER")
dbname = os.getenv("DB_NAME")

db_url = f"{scheme}://{user}:{db_password}@{host}:{port}/{dbname}"
engine = create_engine(db_url)
reader = DatabaseReader(engine=engine)




In [105]:
# 데이터 로드
query = "SELECT * FROM users"

In [106]:
documents = reader.load_data(query=query)

In [107]:
for idx, doc in enumerate(documents):
    print(f"Document {idx + 1 }:")
    print(f"ID {doc.id_}")
    print(f"Row {doc.text}")
    print("-" * 50)

Document 1:
ID 6e0e6320-aae5-487a-8530-8b8468fd28c0
Row id: 1, name: Alice, email: alice@example.com, age: 30
--------------------------------------------------
Document 2:
ID ff38f595-9cd5-4913-a0c8-b66e42f3a894
Row id: 2, name: Bob, email: bob@example.com, age: 25
--------------------------------------------------
Document 3:
ID 6c530f04-f7a1-41ee-828a-6f4eb894baa8
Row id: 3, name: Charlie, email: charlie@example.com, age: 35
--------------------------------------------------


### 2.3 텍스트 분할 
* 목적: 문서를 효과적으로 관리하고 검색
* => 그러기 위해선,, 텍스트를 적절한 크기로 분할 
* => 장점1. 원하는 정보를 더 빠르게 찾을 수 있다.
* => 장점2. 검색 및 인덱싱 성능도 향상..
* 라마인덱스는? => 문서를 여러 개의 노드로 분할하여 저장하고 분석하도록 설계.

### 2.3.1 문서와 노드
* 문서: 원시 데이터를 처리 가능한 형태로 변환한 데이터의 기본 단위
* 노드: 문서를 더 작은 단위로 세분화하여 검색 및 분석할 수 있는 기본 단위.

### 문서(Document) 

In [120]:
from llama_index.core import Document

# 텍스트 데이터를 기반으로 문서 생성.
document_text = (
    "영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 벌어지는 이야기 입니다. "
    "처음에는 평화로워 보이지만, 이들의 거짓말이 쌓이며 긴장감이 점점 고조됩니다. "
    "박 사장네 집에는 비밀 지하실이 존재하며, 그곳에는 오랫동안 숨어 살던 남자가 있다는 반전이 있습니다."
    "이 사실을 알게 된 기우네 가족은 예상치 못한 위기를 맞이하게 됩니다. "
    "결국 극한 상황에서 벌어지는 사건으로 인해 비극적인 결말로 이어집니다."
)
document = Document(text=document_text)
# 메타데이터 추가
document.metadata = {'author': '영화 해설', 'subject': '기생충 줄거리'}

print(document)

Doc ID: 665845b5-43dc-4337-9d8e-f2c01ea6bab4
Text: 영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 벌어지는 이야기 입니다. 처음에는
평화로워 보이지만, 이들의 거짓말이 쌓이며 긴장감이 점점 고조됩니다. 박 사장네 집에는 비밀 지하실이 존재하며, 그곳에는
오랫동안 숨어 살던 남자가 있다는 반전이 있습니다.이 사실을 알게 된 기우네 가족은 예상치 못한 위기를 맞이하게 됩니다. 결국
극한 상황에서 벌어지는 사건으로 인해 비극적인 결말로 이어집니다.


#### 노드(Node)
##### 노드란? 문서를 더 작은 단위로 나눈 데이터 요소 (개별적이고 독립적인 정보 단위)
###### 예) 텍스트 청크, 이미지 패치
###### * 각 노드는 독자적인 메타데이터를 가질 수 있다.



In [111]:
from llama_index.core.node_parser import SimpleNodeParser # 문서를 여러개의 노드로 쉽게 분할

parser = SimpleNodeParser(chunk_size=80, chunk_overlap=0)
nodes = parser.get_nodes_from_documents([document])

print("\n생성된 노드들")

for idx, node in enumerate(nodes, start=1):
    node.metadata = {'type': '영화 줄거리', 'genre': '드라마', 'node_id': idx}
    print(f"노드 {idx}: {node.text}")
    print(f"    메타데이터: {node.metadata}")


생성된 노드들
노드 1: 영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 벌어지는 이야기 입니다.
    메타데이터: {'type': '영화 줄거리', 'genre': '드라마', 'node_id': 1}
노드 2: 처음에는 평화로워 보이지만, 이들의 거짓말이 쌓이며 긴장감이 점점 고조됩니다.
    메타데이터: {'type': '영화 줄거리', 'genre': '드라마', 'node_id': 2}
노드 3: 박 사장네 집에는 비밀 지하실이 존재하며, 그곳에는 오랫동안 숨어 살던 남자가 있다는 반전이 있습니다.
    메타데이터: {'type': '영화 줄거리', 'genre': '드라마', 'node_id': 3}
노드 4: 이 사실을 알게 된 기우네 가족은 예상치 못한 위기를 맞이하게 됩니다.
    메타데이터: {'type': '영화 줄거리', 'genre': '드라마', 'node_id': 4}
노드 5: 결국 극한 상황에서 벌어지는 사건으로 인해 비극적인 결말로 이어집니다.
    메타데이터: {'type': '영화 줄거리', 'genre': '드라마', 'node_id': 5}


* 청킹의 기본 단위: 토큰 (Token)
* 토큰: 텍스트를 작은 단위로 분할한 결과물
* 예: 단어, 형태소(부분 단어), 문자 단위 
* 분할 과정에는 토크나이저(Tokenizer) 사용
* 토크나이저: 문장을 일정한 규칙에 따라 토큰으로 변환하는 역할.

In [121]:
# 지정한 텍스트를 80 토큰 단위로 나누는 과정 
# TokenTextSplitter

from llama_index.core.node_parser import TokenTextSplitter
import tiktoken

# cl100k_base 토크나이저 로드
tokenizer = tiktoken.get_encoding("cl100k_base")

token_splitter = TokenTextSplitter(chunk_size=80, chunk_overlap=0)

chunks = token_splitter.split_text(document_text)

for i, chunk in enumerate(chunks):
    token_count = len(tokenizer.encode(chunk)) # 각 청크의 토큰 개수 계산 
    print(f"청크 {i+1}: {chunk}\n[토큰 개수: {token_count}]\n")

print(f"총 생성된 청크 개수: {len(chunks)}")

청크 1: 영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 벌어지는 이야기 입니다. 처음에는 평화로워 보이지만, 이들의
[토큰 개수: 78]

청크 2: 거짓말이 쌓이며 긴장감이 점점 고조됩니다. 박 사장네 집에는 비밀 지하실이 존재하며, 그곳에는 오랫동안 숨어 살던 남자가 있다는
[토큰 개수: 78]

청크 3: 반전이 있습니다.이 사실을 알게 된 기우네 가족은 예상치 못한 위기를 맞이하게 됩니다. 결국 극한 상황에서 벌어지는 사건으로 인해 비극적인 결말로 이어집니다.
[토큰 개수: 80]

총 생성된 청크 개수: 3


### 노드 단위 검색의 장점

* 노드 단위 검색 활용: 검색 속도, 정확도가 크게 향상된다. 

* chunk_overlap: 각 청크별로 문자열을 얼마나 겹칠지를 결정 (0으로 설정: 각 청크가 서로 겹치지 않도록 청킹한다는 의미, 빠르고 정밀한 검색 결과 제공 가능.)

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.llms.openai import OpenAI

import os

# OpenAI 설정
llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"], model="gpt-4o-mini",
             system_prompt="반드시 한국어로 답변하세요.")

# 문서 단위 검색을 위한 전체 문서 인덱스 생성
full_doc_index = VectorStoreIndex.from_documents([document], llm=llm)

# 노드 단위 검색을 위한 인덱스 생성
node_index=VectorStoreIndex(nodes, llm=llm)

# 검색 비교
query_text = '이 영화의 반전은?'

## 문서 단위 검색
print("\n문서 단위 검색 결과")
doc_query_engine = full_doc_index.as_query_engine()
doc_response = doc_query_engine.query(query_text)
print(f"문서 검색 응답: {doc_response.response}")
if doc_response.source_nodes:
    for idx, document in enumerate(doc_response.source_nodes, start=1):
        print(f" - 결과 {idx}: {document.node.text}")




문서 단위 검색 결과
문서 검색 응답: The movie's twist reveals the existence of a secret basement in the wealthy Park family's house, where a man has been hiding for a long time. This unexpected revelation leads to a tragic outcome for the poor Kim family.
 - 결과 1: 영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 벌어지는 이야기 입니다. 처음에는 평화로워 보이지만, 이들의 거짓말이 쌓이며 긴장감이 점점 고조됩니다. 박 사장네 집에는 비밀 지하실이 존재하며, 그곳에는 오랫동안 숨어 살던 남자가 있다는 반전이 있습니다.이 사실을 알게 된 기우네 가족은 예상치 못한 위기를 맞이하게 됩니다. 결국 극한 상황에서 벌어지는 사건으로 인해 비극적인 결말로 이어집니다.


In [ ]:
## 노드 단위 검색
print("\n노드 단위 검색 결과")
node_query_engine = node_index.as_query_engine()
node_response=node_query_engine.query(query_text)


노드 단위 검색 결과


In [124]:
print(f"노드 검색 응답: {node_response.response}")

노드 검색 응답: 남자가 박 사장네 집의 비밀 지하실에 숨어 살고 있다는 반전입니다.


In [125]:
if node_response.source_nodes:
    for idx,document in enumerate(node_response.source_nodes, start=1):
        print(f"- 결과 노드 {idx}: {document.node.text}")

- 결과 노드 1: 박 사장네 집에는 비밀 지하실이 존재하며, 그곳에는 오랫동안 숨어 살던 남자가 있다는 반전이 있습니다.
- 결과 노드 2: 결국 극한 상황에서 벌어지는 사건으로 인해 비극적인 결말로 이어집니다.


* 노드 단위 검색의 장점: 문서를 의미 단위로 나눈 각 청크에 직접 접근 가능
* => 문서 전체를 일일이 탐색하지 않고도 질의와 관련된 핵심 정보 빠르게 찾을 수 있음.
* => 결론: 검색 시간 단축, 정확도 높아짐.

In [127]:
# 기본 청킹 크기: 1024
from llama_index.core.node_parser import SimpleNodeParser
parser = SimpleNodeParser()
print(f"기본 청킹 크기: {parser.chunk_size}")

기본 청킹 크기: 1024


* 청크가 더 클수록? : 불필요한 정보까지 포함하여 검색 정확도를 떨어뜨릴 수 있음.
* 청크가 엄청 작을수록? : 문맥이 단절되어 핵심 정보 파악 어려움.

* 평균 신뢰도: 생성된 응답이 원본 문서(출처)와 얼마나 일치하는지 평가
* 평균 관련성: 생성된 응답이 사용자 질문과 얼마나 관련이 있는지 평가

* 응답시간과 응답 품질(신뢰도 및 관련성)의 균형 맞추기엔 1,024 값이 좋다.

###

### 2.3.2 토큰 단위 분할
* 문서를 일정한 길이의 토큰(token) 단위로 나누는 방식
* 토큰: 단어나 구두점 등 텍스트를 구성하는 기본적인 단위를 의미.
* 토큰 단위로 나누면,, 문서를 일정한 크기의 블록으로 나눌 수 있어 전체 문서의 길이가 일정하게 유지됨.
* 장점: 메모리 관리가 편함. 
* 단점: 문장이 중간에서 인위적으로 잘릴 수 있음, 문맥이 단절되거나 원래 의미가 훼손될 수 있음.

In [ ]:
sample_text = "영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 벌어지는 이야기입니다. 처음에는 평화로워 보이지만, 이들의 거짓말이 쌓이며 긴장감이 점점 고조됩니다. 그러나 이 영화의 반전은 지하실에서 시작됩니다."

token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10 # 각 청크가 앞 뒤로 10토큰씩 중복되도록 설정.
)

token_chunks = token_splitter.split_text(sample_text)
print("=== 토큰 기반 분할 결과 ===")
for i, chunk in enumerate(token_chunks):
    print(f"Chunk {i+1}:", chunk.strip(), "\n")

=== 토큰 기반 분할 결과 ===
Chunk 1: 영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 

Chunk 2: 취업하면서 벌어지는 이야기입니다. 처음에는 평화로워 보이지만, 이들의 거짓말이 쌓이며 

Chunk 3: 쌓이며 긴장감이 점점 고조됩니다. 그러나 이 영화의 반전은 지하실에서 시작됩니다. 



### 2.3.3 문장 단위 분할
* 문서를 각 문장을 기준으로 나누는 방식.
* 장점: 토큰 단위 분할보다 문장이 불완전하게 나뉠 가능성이 낮으며, 문맥의 흐름을 보다 자연스럽게 유지할 수 있음.
* 장점2: 문맥의 일관성을 유지하는 데 유리, 토큰 단위 분할 보다 문장이 불완전하게 잘리는 현상을 최소화할 수 있는 장점 있음.

In [133]:
from llama_index.core.node_parser.text.sentence import SentenceSplitter

splitter = SentenceSplitter(chunk_size=50, chunk_overlap=0)

sentence_chunks=splitter.split_text(sample_text)
print("=== 문장 기반 분할 결과 ===")
for i, chunk in enumerate(sentence_chunks):
    print(f"chunk {i+1}:", chunk.strip(), "\n")

=== 문장 기반 분할 결과 ===
chunk 1: 영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 

chunk 2: 벌어지는 이야기입니다. 

chunk 3: 처음에는 평화로워 보이지만, 이들의 거짓말이 쌓이며 긴장감이 점점 고조됩니다. 

chunk 4: 그러나 이 영화의 반전은 지하실에서 시작됩니다. 



### 2.3.4 의미 단위 분할 (Semantic Splitting) 
* 문맥의 의미를 고려하여 텍스트를 적절한 단위로 분할하는 방식

* 임베딩을 활용하여 텍스트의 의미적 유사도를 계산, 의미가 자연스럽게 유지되는 지점에서 텍스트를 분할하는 것이 특징.

In [142]:

from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding 
# OpenAIeMBEDDING 활용 
# => 텍스트의 의미적 유사도 계산, 문맥상 부자연스럽게 끊기지 않도록 전후 의미 흐름 분석, 의미 단절 최소화되는 지점을 분할 지점으로 선택.
# 결론: 의미 단위가 자연스럽게 구분되는 위치에서 청크를 나누어 중요한 문맥을 유지.

embed_model = OpenAIEmbedding()

splitter = SemanticSplitterNodeParser(
    buffer_size=1, # 분할 시 인접 문장들을 함께 고려하는 범위를 의미.
    breakpoint_percentile_threshold=95,  # 의미적으로 중요한 분할 지점을 결정하는 임계값. 값 낮게 설정하면 작은 의미 변화에도 민감 하게 반응, 더 자주 분할 발생.
    embed_model=embed_model
)

chunks=splitter.sentence_splitter(sample_text)
print("=== 의미 기반 분할 결과 ===")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:", chunk.strip(), "\n")



=== 의미 기반 분할 결과 ===
Chunk 1: 영화 '기생충'은 가난한 가족인 기우네가 부유한 박 사장네 집에 하나씩 취업하면서 벌어지는 이야기입니다. 

Chunk 2: 처음에는 평화로워 보이지만, 이들의 거짓말이 쌓이며 긴장감이 점점 고조됩니다. 

Chunk 3: 그러나 이 영화의 반전은 지하실에서 시작됩니다. 



* 위 청크의 의미 분석:
* 1. 영화의 기본적인 소개 포함.
* 2. 이야기의 전개를 다룸.
* 3. 반전이라는 주제 중심 서술.

* 장점: 사용자의 질문 의도에 부합하는 응답을 생성하는데 더욱 효과적.